In [ ]:
# Passo 0: Instalação de Dependências
!pip install google-adk -q
!pip install litellm -q
!pip install python-dotenv -q

print("Instalação concluída.")

In [1]:
# Importar bibliotecas necessárias
import os
import asyncio
from dotenv import load_dotenv
from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.sessions import InMemorySessionService
from google.adk.runners import Runner
from google.genai import types

import warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.ERROR)

print("Bibliotecas importadas.")

Bibliotecas importadas.


In [32]:
# Configurar API Keys
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"

# Definir modelos
MODEL_GEMINI_2_5_FLASH = "gemini-2.5-flash"
MODEL_LLAMA_GROQ = "groq/llama-3.3-70b-versatile"

print("API Keys Set:")
print(f"Google API Key set: {'Yes' if os.environ.get('GOOGLE_API_KEY') and os.environ['GOOGLE_API_KEY'] != 'YOUR_GOOGLE_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")
print(f"Groq API Key set: {'Yes' if os.environ.get('GROQ_API_KEY') and os.environ['GROQ_API_KEY'] != 'YOUR_GROQ_API_KEY' else 'No (REPLACE PLACEHOLDER!)'}")

API Keys Set:
Google API Key set: Yes
Groq API Key set: Yes


## Passo 1: Definir Ferramentas dos Agentes

In [3]:
# Ferramenta 1: Pesquisar notícias/tópicos
def buscar_noticias(topico: str) -> dict:
    """Busca informações sobre um tópico jornalístico.
    
    Args:
        topico (str): Tópico a pesquisar (eleições, tecnologia, meio ambiente, economia)
    
    Returns:
        dict: Dicionário com status, resumo, fontes e data
    """
    print(f"--- Ferramenta: buscar_noticias chamada para tópico: {topico} ---")
    
    topico_normalizado = topico.lower().replace(" ", "")
    
    # Base de dados mock de notícias
    noticias_db = {
        "eleições": { 
            "status": "sucesso",
            "resumo": "Últimas eleições presidenciais realizadas em diversos países com participação acima de 70%. Campanhas focaram em economia, educação e políticas climáticas. Resultados mostram maior engajamento de eleitores jovens. Líderes eleitos prometem reformas estruturais nos próximos 100 dias de governo."
        },
        "tecnologia": { 
            "status": "sucesso",
            "resumo": "Avanços revolucionários em inteligência artificial generativa e computação quântica dominam mercado tecnológico. Empresas lançam modelos de IA com 40% mais eficiência energética. Computadores quânticos alcançam 1000 qubits estáveis. Investimentos globais em IA ultrapassam 500 bilhões de dólares. Regulamentação de IA avança em discussões internacionais."
        },
        "meioambiente": { 
            "status": "sucesso",
            "resumo": "Conferência climática global estabelece metas ambiciosas: redução de 60% em emissões até 2030. 195 países assinam novo acordo vinculante. Foco em energia renovável, reflorestamento e transição industrial. Brasil lidera iniciativa de proteção da Amazônia. Investimentos em tecnologia verde totalizam 2 trilhões de dólares. Cientistas alertam que janela de oportunidade se fecha."
        },
        "economia": { 
            "status": "sucesso",
            "resumo": "Bancos Centrais ao redor do mundo aumentam taxas de juros para combater inflação elevada. Mercados reagem com volatilidade: ações caem 8%, moedas emergentes sofrem pressão. Desemprego permanece em 5.2% globalmente. Preços de commodities continuam elevados. Expectativas de recessão moderada em 2026. Investidores buscam ativos seguros como ouro e títulos do governo."
        }
    }
    
    if topico_normalizado in noticias_db:
        return noticias_db[topico_normalizado]
    else:
        return {
            "status": "error",
            "error_message": f"Tópico '{topico}' não encontrado."
        }
        

# Teste da ferramenta
print(buscar_noticias("tecnologia"))
print(buscar_noticias("esportes"))


--- Ferramenta: buscar_noticias chamada para tópico: tecnologia ---
{'status': 'sucesso', 'resumo': 'Avanços revolucionários em inteligência artificial generativa e computação quântica dominam mercado tecnológico. Empresas lançam modelos de IA com 40% mais eficiência energética. Computadores quânticos alcançam 1000 qubits estáveis. Investimentos globais em IA ultrapassam 500 bilhões de dólares. Regulamentação de IA avança em discussões internacionais.'}
--- Ferramenta: buscar_noticias chamada para tópico: esportes ---
{'status': 'error', 'error_message': "Tópico 'esportes' não encontrado."}


In [22]:
# Ferramenta 2: Verificar fatos
def verificar_fatos(afirmacao: str) -> dict:
    """Verifica a precisão de uma afirmação.
    
    Args:
        afirmacao (str): Afirmação a verificar
    
    Returns:
        dict: Resultado da verificação (verdadeiro, falso ou indeterminado)
    """
    print(f"--- Ferramenta: verificar_fatos chamada para: {afirmacao} ---")
    
    afirmacao_normalizado = afirmacao.lower()
    
    fatos_verificados = {
        # Verificações sobre eleições
        "eleições aumentaram engajamento de jovens": {"resultado": "VERDADEIRO", "confiança": 87},
        "participação eleitoral foi acima de 70%": {"resultado": "VERDADEIRO", "confiança": 85},
        "governos eleitos têm 100 dias para reformas": {"resultado": "VERDADEIRO", "confiança": 90},
        # Verificações sobre tecnologia
        "inteligência artificial melhorou eficiência energética": {"resultado": "VERDADEIRO", "confiança": 92},
        "computadores quânticos atingiram 1000 qubits": {"resultado": "VERDADEIRO", "confiança": 88},
        "investimentos em IA ultrapassam 500 bilhões": {"resultado": "VERDADEIRO", "confiança": 84},
        # Verificações sobre meio ambiente
        "195 países assinaram novo acordo climático": {"resultado": "VERDADEIRO", "confiança": 91},
        "meta é reduzir emissões em 60% até 2030": {"resultado": "VERDADEIRO", "confiança": 89},
        "brasil lidera proteção da amazônia": {"resultado": "VERDADEIRO", "confiança": 86},
        # Verificações sobre economia
        "bancos centrais aumentaram taxas de juros": {"resultado": "VERDADEIRO", "confiança": 93},
        "ações caíram 8% recentemente": {"resultado": "VERDADEIRO", "confiança": 85},
        "desemprego global está em 5.2%": {"resultado": "VERDADEIRO", "confiança": 83},
    }
    
    if afirmacao_normalizado in fatos_verificados:
            return fatos_verificados[afirmacao_normalizado]
    else: 
        return {
            "status": "error",
            "error_message": f"Afirmação '{afirmacao}' não conseguiu ser verificada."
        }

# Teste da ferramenta
print(verificar_fatos("Eleições aumentaram engajamento de jovens"))
print(verificar_fatos("Marte é um planeta"))

--- Ferramenta: verificar_fatos chamada para: Eleições aumentaram engajamento de jovens ---
{'resultado': 'VERDADEIRO', 'confiança': 87}
--- Ferramenta: verificar_fatos chamada para: Marte é um planeta ---
{'status': 'error', 'error_message': "Afirmação 'Marte é um planeta' não conseguiu ser verificada."}


## Passo 2: Definir Agentes Auxiliadores

In [23]:
# Agente Auxiliador 1: Pesquisador de Notícias
pesquisador_agent = Agent(
    name="pesquisador_noticias",
    model=LiteLlm(model=MODEL_LLAMA_GROQ),
    description="Especialista em pesquisa de notícias e tópicos jornalísticos.",
    instruction="Você é um pesquisador de notícias. Use a ferramenta 'buscar_noticias' para encontrar "
                "informações sobre tópicos solicitados. Apresente os resultados de forma clara.",
    tools=[buscar_noticias],
)
print(f"✅ Agente '{pesquisador_agent.name}' criado com modelo {MODEL_LLAMA_GROQ}.")

# Agente Auxiliador 2: Verificador de Fatos  
verificador_agent = Agent(
    name="verificador_fatos",
    model=LiteLlm(model=MODEL_LLAMA_GROQ),
    description="Especialista em verificação de fatos e análise crítica.",
    instruction="Você é um fact-checker. Use a ferramenta 'verificar_fatos' para validar afirmações. "
                "Indique o resultado e confiança de cada verificação."
                "Se a ferramenta não tiver a verificação, informe que não tem certeza da resposta.",
    tools=[verificar_fatos],
)
print(f"✅ Agente '{verificador_agent.name}' criado com modelo {MODEL_LLAMA_GROQ}.")

✅ Agente 'pesquisador_noticias' criado com modelo groq/llama-3.3-70b-versatile.
✅ Agente 'verificador_fatos' criado com modelo groq/llama-3.3-70b-versatile.


## Passo 3: Definir Agente Orquestrador

In [24]:
# Agente Orquestrador Principal
agente_orquestrador = Agent(
    name="editor_jornalistico",
    model=MODEL_GEMINI_2_5_FLASH,
    description="Coordena pesquisas de notícias e verificação de fatos.",
    instruction="Você é um editor jornalístico assistido por IA. Coordene as tarefas de pesquisa e verificação. "
                "Para pesquisar tópicos, delegue ao 'pesquisador_noticias'. "
                "Para verificar fatos, delegue ao 'verificador_fatos'. "
                "Resuma os resultados de forma clara.",
    tools=[],
    sub_agents=[pesquisador_agent, verificador_agent],
    output_key="resposta_jornalistica"
)

print(f"✅ Agente '{agente_orquestrador.name}' criado com modelo {MODEL_GEMINI_2_5_FLASH}.")

✅ Agente 'editor_jornalistico' criado com modelo gemini-2.5-flash.


## Passo 4: Configurar Sessão e Runner

In [25]:
# Configurar Sessão e Runner

session_service = InMemorySessionService()

APP_NAME = "time_jornalista"
USER_ID = "jornalista_1"
SESSION_ID = "sessao_001"

session = await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID
)
print(f"Sessão criada: App='{APP_NAME}', User='{USER_ID}', Session='{SESSION_ID}'")

runner = Runner(
    agent=agente_orquestrador,
    app_name=APP_NAME,
    session_service=session_service
)
print(f"Runner criado para '{runner.agent.name}'.")

Sessão criada: App='time_jornalista', User='jornalista_1', Session='sessao_001'
Runner criado para 'editor_jornalistico'.


## Passo 5: Função de Interação

In [26]:
# Função para consultar o agente
async def consultar_agente(pergunta: str, runner, user_id, session_id):
    """Consulta o agente com uma pergunta."""
    print(f"\n>>> Pergunta: {pergunta}")
    
    content = types.Content(role='user', parts=[types.Part(text=pergunta)])
    
    resposta_final = "Agente não produziu resposta."
    
    async for event in runner.run_async(user_id=user_id, session_id=session_id, new_message=content):
        if event.is_final_response():
            if event.content and event.content.parts:
                resposta_final = event.content.parts[0].text
            break
    
    print(f"<<< Resposta: {resposta_final}")

## Passo 6: Testar o Sistema

In [28]:
# Teste 1: Pesquisa de notícias
print("\n--- Teste 1: Pesquisa de Notícias ---")

await consultar_agente(
    "Pesquise informações sobre tecnologia para meu artigo.",
    runner=runner,
    user_id=USER_ID,
    session_id=SESSION_ID
)


--- Teste 1: Pesquisa de Notícias ---

>>> Pergunta: Pesquise informações sobre tecnologia para meu artigo.
--- Ferramenta: buscar_noticias chamada para tópico: tecnologia ---
<<< Resposta: Após realizar uma pesquisa sobre tecnologia, encontrei as seguintes informações que podem ser úteis para o seu artigo:

* Avanços revolucionários em inteligência artificial generativa e computação quântica dominam o mercado tecnológico.
* Empresas lançam modelos de IA com 40% mais eficiência energética.
* Computadores quânticos alcançam 1000 qubits estáveis.
* Investimentos globais em IA ultrapassam 500 bilhões de dólares.
* Regulamentação de IA avança em discussões internacionais.

Essas informações podem ser úteis para fornecer uma visão geral dos principais desenvolvimentos e tendências atuais no setor de tecnologia. Se precisar de mais ajuda ou tiver outras perguntas, não hesite em perguntar.


In [29]:
# Teste 2: Verificação de fatos
print("\n--- Teste 2: Verificação de Fatos ---")

await consultar_agente(
    "Preciso tentar verificar se 'bancos centrais aumentaram taxas de juros' está correto.",
    runner=runner,
    user_id=USER_ID,
    session_id=SESSION_ID
)


--- Teste 2: Verificação de Fatos ---

>>> Pergunta: Preciso tentar verificar se 'bancos centrais aumentaram taxas de juros' está correto.
--- Ferramenta: verificar_fatos chamada para: bancos centrais aumentaram taxas de juros ---
<<< Resposta: A afirmação 'bancos centrais aumentaram taxas de juros' foi verificada e o resultado indica que é VERDADEIRO com uma confiança de 93%. Isso significa que, com base nas informações disponíveis, é provável que os bancos centrais tenham aumentado as taxas de juros, mas é importante considerar que a confiança não é de 100%, deixando um pequeno espaço para incertezas ou variações dependendo do contexto específico ou do momento em que a verificação foi feita.


In [33]:
# Teste 3: Pesquisa + Verificação
print("\n--- Teste 3: Pesquisa + Verificação ---")
await consultar_agente(
    "Pesquise sobre eleições e verifique se 'eleições aumentaram engajamento de jovens'",
    runner=runner,
    user_id=USER_ID,
    session_id=SESSION_ID
)


--- Teste 3: Pesquisa + Verificação ---

>>> Pergunta: Pesquise sobre eleições e verifique se 'eleições aumentaram engajamento de jovens'
--- Ferramenta: verificar_fatos chamada para: eleições aumentaram engajamento de jovens ---
--- Ferramenta: buscar_noticias chamada para tópico: eleições ---
<<< Resposta: Após realizar uma pesquisa sobre eleições, encontrei as seguintes informações que podem ser úteis para o seu artigo:

* Últimas eleições presidenciais realizadas em diversos países com participação acima de 70%.
* Campanhas focaram em economia, educação e políticas climáticas.
* Resultados mostram maior engajamento de eleitores jovens.
* Líderes eleitos prometem reformas estruturais nos próximos 100 dias de governo.

Essas informações podem ser úteis para fornecer uma visão geral dos principais desenvolvimentos e tendências atuais no setor de eleições. Além disso, a afirmação "eleições aumentaram engajamento de jovens" foi verificada e o resultado indica que é VERDADEIRO com uma c